In [1]:
import logging
import os
import time
import zipfile

import requests
from minio import Minio
from pyspark.sql import SparkSession, types
from pyspark.sql import functions as F

In [2]:
client = Minio(
    "minio:9000", access_key="minioadmin", secret_key="minioadmin", secure=False
)
bucket = "crypto-data-lake"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [19]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def download_file(url, file_name):
    if os.path.exists(file_name):
        logger.info(f"{file_name} exits")
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(file_name, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        logger.info(
            f"Downloaded {file_name} {(os.path.getsize(file_name) / (1024 * 1024)):.2f}MB completed"
        )


def remove_file(file_name):
    if os.path.exists(file_name):
        os.remove(file_name)
        logger.info(f"{file_name} removed")
    else:
        logger.info(f"{file_name} not found")


def extract_file(extract_dir, zip_path):
    if not os.path.exists(zip_path):
        logger.info(f"{zip_path} not found")
        return
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
        csv_file = os.path.join(extract_dir, os.listdir(extract_dir)[0])
        logger.info(f"Extracted CSV: {csv_file}")
        return csv_file

In [46]:
extract_dir = "unzipped_data"
landing_date = "2025-08-05"
symbol = "ETHUSDT"
url = f"https://data.binance.vision/data/spot/daily/aggTrades/{symbol}/{symbol}-aggTrades-{landing_date}.zip"
file_name = url.split("/")[-1]
logger.info(file_name)

INFO:__main__:ETHUSDT-aggTrades-2025-08-05.zip


In [47]:
start_t = time.time()
download_file(url, file_name)
end_t = time.time()
logger.info(f"Processed in {(end_t - start_t):.3f} seconds")

INFO:__main__:Downloaded ETHUSDT-aggTrades-2025-08-05.zip 18.32MB completed
INFO:__main__:Processed in 2.074 seconds


In [48]:
csv_file = extract_file(extract_dir, file_name)

INFO:__main__:Extracted CSV: unzipped_data/ETHUSDT-aggTrades-2025-08-05.csv


In [22]:
spark = SparkSession.builder \
    .appName("LandingZone") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .getOrCreate()

25/10/04 14:31:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [32]:
schema = types.StructType(
    [
        types.StructField("agg_trade_id", types.LongType(), True),
        types.StructField("price", types.DoubleType(), True),
        types.StructField("quantity", types.DoubleType(), True),
        types.StructField("first_trade_id", types.LongType(), True),
        types.StructField("last_trade_id", types.LongType(), True),
        types.StructField("timestamp", types.LongType(), True),
        types.StructField("is_buyer_maker", types.BooleanType(), True),
        types.StructField("is_best_match", types.BooleanType(), True),
    ]
)

In [49]:
df = spark.read.option("header", "false").schema(schema).csv(csv_file)

In [50]:
df = df.withColumn("ingest_date", F.current_date()).withColumn(
    "ingest_timestamp", F.current_timestamp()
)

In [51]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/{symbol}/{landing_date}"
df.write.mode("overwrite").parquet(output_path)
logger.info(f"Parquet written to: {output_path}")

INFO:__main__:Parquet written to: s3a://crypto-data-lake/landing_zone/spot/daily/aggTrades/ETHUSDT/2025-08-05


In [52]:
remove_file(csv_file)
remove_file(file_name)

INFO:__main__:unzipped_data/ETHUSDT-aggTrades-2025-08-05.csv removed
INFO:__main__:ETHUSDT-aggTrades-2025-08-05.zip removed


In [53]:
!jupyter nbconvert --to script end_landing_job.ipynb

IOStream.flush timed out
[NbConvertApp] Converting notebook end_landing_job.ipynb to script
[NbConvertApp] Writing 3788 bytes to end_landing_job.py
